# Protobuf Basics — 10: Protobuf Best Practices

Capstone notebook: complete reference for production Protobuf usage,
common pitfalls, tooling setup, and integration with Spark 4.0.2.

Topics: .proto design guidelines, descriptor management, Spark integration
checklist, complete pipeline, format selection summary.


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/protobuf_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('protobuf-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — .proto Design Guidelines



In [ ]:

print("""
.proto design guidelines:

FIELD NUMBERS:
  - 1-15: use for the most frequently populated fields (1-byte tag)
  - 16-2047: use for less common fields (2-byte tag)
  - Assign in order of importance, not alphabetically
  - NEVER reuse a field number

NAMING CONVENTIONS:
  - Message names: PascalCase (Order, LineItem, UserProfile)
  - Field names: snake_case (order_id, customer_email, created_at)
  - Enum values: SCREAMING_SNAKE_CASE prefixed with type name
    OrderStatus.ORDER_STATUS_PENDING  (prefix avoids global collision)
  - File names: snake_case (order_events.proto)

TYPES:
  - Use int64 for IDs (not int32 — IDs grow over time)
  - Use double for money amounts (or use cents as int64 for exactness)
  - Use google.protobuf.Timestamp for timestamps (not string)
  - Use optional scalar fields (proto3 optional) when 0/false is ambiguous

STRUCTURE:
  - Keep messages focused (single responsibility)
  - Avoid deeply nested messages (3 levels max)
  - Use separate message for repeated complex types
  - Add doc comments (// or /* */) to every field
""")


## Step 2 — Descriptor File Management



In [ ]:

print("""
Descriptor file (.desc) management in production:

  1. Check .proto files into version control
     git/orders.proto
     git/users.proto

  2. Compile descriptors in CI/CD pipeline
     protoc --descriptor_set_out=orders.desc \
            --include_imports orders.proto

  3. Store compiled descriptors in accessible location
     - HDFS/S3: hdfs:///schema/protobuf/orders_v1.desc
     - Local: /workspace/schemas/orders.desc
     - Schema Registry (Confluent): register via API

  4. Version your descriptor files
     orders_v1.desc  ← never mutate
     orders_v2.desc  ← new file for each breaking change

  5. Reference in Spark code
     from pyspark.sql.protobuf.functions import from_protobuf
     DESC = "/workspace/schemas/orders_v2.desc"
     df.select(from_protobuf(col("bytes"), "Order", DESC).alias("o"))
""")


## Step 3 — Spark 4.0.2 Protobuf Checklist



In [ ]:

import pathlib, subprocess

print("Spark 4.0.2 Protobuf Integration Checklist:")
print()

# 1. Check spark-protobuf availability
try:
    from pyspark.sql.protobuf.functions import from_protobuf, to_protobuf
    print("  ✅ pyspark.sql.protobuf.functions available (Spark 3.4+)")
except ImportError:
    print("  ❌ from_protobuf not available — needs spark-protobuf JAR")

# 2. Check protobuf Python library
try:
    import google.protobuf
    print(f"  ✅ protobuf Python library: {google.protobuf.__version__}")
except ImportError:
    print("  ❌ protobuf Python library not installed (pip install protobuf)")

# 3. Check grpcio-tools
r = subprocess.run(["python3","-m","grpc_tools.protoc","--version"],
                   capture_output=True, text=True)
if r.returncode == 0:
    print(f"  ✅ grpcio-tools available (protoc compiler)")
else:
    print("  ⚠️  grpcio-tools not installed (pip install grpcio-tools)")

# 4. Check descriptor file
desc_file = f"{DATA_DIR}/protos/orders.desc"
if pathlib.Path(desc_file).exists():
    print(f"  ✅ Descriptor file: {desc_file}")
else:
    print(f"  ⚠️  No descriptor file at {desc_file}")
    print("     Run: python -m grpc_tools.protoc --descriptor_set_out=orders.desc orders.proto")


## Step 4 — Complete Pipeline Recap



In [ ]:

print("""
Complete Protobuf data pipeline recap:

  1. Define schema (.proto file)
     syntax = "proto3";
     message Order { int64 order_id = 1; string product = 2; ... }

  2. Compile to descriptor
     python -m grpc_tools.protoc --descriptor_set_out=orders.desc --include_imports orders.proto

  3. Produce Protobuf messages (Python)
     import orders_pb2
     o = orders_pb2.Order(); o.order_id = 1; o.product = "Laptop"
     kafka_producer.send("orders", o.SerializeToString())

  4. Consume in Spark (batch or streaming)
     from pyspark.sql.protobuf.functions import from_protobuf
     df.select(from_protobuf(col("value"), "Order", "orders.desc").alias("o"))
       .select("o.*")

  5. Write to Parquet/Delta for analytics
     df.write.format("delta").option("path","/data/orders/").save()
""")


## Step 5 — Protobuf Basics Recap + All Formats Summary



In [ ]:

print("""
Protobuf Basics — 10 notebooks:
  01 what_is_protobuf      — format overview, gRPC use case, Spark integration
  02 proto_schema          — .proto syntax, types, nested messages, field numbers
  03 serialization         — wire format, Python serialize/deserialize, size comparison
  04 spark_protobuf        — from_protobuf, to_protobuf, descriptor files
  05 schema_evolution      — field number rules, reserved, compatibility vs Avro
  06 protobuf_kafka        — Kafka + Protobuf pattern, Confluent SR, streaming
  07 protobuf_vs_json_avro — comprehensive comparison, decision guide
  08 nested_protobuf       — struct/array/map in Spark, flattening
  09 protobuf_to_parquet   — complete pipeline, UDF deserialization, validation
  10 best_practices        — design guidelines, descriptor management, checklist

Complete basics/ folder summary:
  csv/      → text format, universal exchange, ETL landing zone
  parquet/  → columnar analytics, the standard for data lakes
  delta/    → ACID transactions, time travel, DML on data lakes
  iceberg/  → open table format, partition evolution, multi-engine
  avro/     → Kafka streaming, schema evolution, row-based binary
  orc/      → Hive ecosystem, bloom filters, 3-level statistics
  json/     → REST APIs, logs, human-readable, flexible schema
  protobuf/ → gRPC services, minimum message size, field numbers
""")
